### Import Libraries

In [1]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

C:\Users\user\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\user\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Extract Dataset

In [2]:
zip_path = "/content/PRAICP-1000-IndiSignLang (1).zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("IndianSignDataset")

print("Dataset Extracted Successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/PRAICP-1000-IndiSignLang (1).zip'

### Set Dataset Path

In [ ]:
dataset_path = "IndianSignDataset/Data"

print(os.listdir(dataset_path))

### Count Images

In [ ]:
classes = os.listdir(dataset_path)

for cls in classes:
    count = len(os.listdir(os.path.join(dataset_path, cls)))
    print(cls, count)

### Display Sample Images

In [ ]:
plt.figure(figsize=(12,8))

for i, cls in enumerate(classes[:9]):
    img_path = os.path.join(dataset_path, cls,
                            os.listdir(os.path.join(dataset_path, cls))[0])

    img = plt.imread(img_path)

    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.title(cls)
    plt.axis("off")

plt.show()

### Data Preprocessing

In [ ]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

### Training Data

In [ ]:
train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(64,64),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

### Validation Data

In [ ]:
validation_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(64,64),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

### uild CNN Model

In [ ]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',input_shape=(64,64,3)))
model.add(MaxPooling2D())

model.add(Conv2D(64,(3,3),activation='relu'))
model.add(MaxPooling2D())

model.add(Conv2D(128,(3,3),activation='relu'))
model.add(MaxPooling2D())

model.add(Flatten())

model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(train_data.num_classes,activation='softmax'))

### Compile Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

### Train Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=20,
    callbacks=[early_stop]
)

### Accuracy Graph

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['accuracy'],label='Training Accuracy')
plt.plot(history.history['val_accuracy'],label='Validation Accuracy')

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Loss Graph

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history['loss'],label='Training Loss')
plt.plot(history.history['val_loss'],label='Validation Loss')

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

### Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(validation_data)

print("Validation Accuracy:", accuracy)

### Save Model

In [ ]:
model.save("IndianSignLanguageModel.h5")

print("Model Saved Successfully")

### Predict New Image

In [ ]:
from tensorflow.keras.preprocessing import image

# Using an existing image from the dataset for demonstration
# You can replace this with any image path from your dataset or an uploaded test.jpg
# For example, using the first image of class 'A'
example_image_path = os.path.join(dataset_path, 'A', os.listdir(os.path.join(dataset_path, 'A'))[0])
img = image.load_img(example_image_path, target_size=(64,64))
img_array = image.img_to_array(img)/255
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

predicted_class = list(train_data.class_indices.keys())[np.argmax(prediction)]

print("Predicted Sign:", predicted_class)